# Notebook 05 — SBERT Embeddings + Hybrid XGBoost

**What this notebook builds:**
1. Encodes all complaint narratives into 384-dim semantic embeddings using `all-MiniLM-L6-v2` (SBERT)
2. Trains XGBoost on **SBERT-only** features — our semantic baseline
3. Trains XGBoost on **TF-IDF + SBERT hybrid** features — our best classical model
4. Compares all three models side by side

**Why SBERT fixes what TF-IDF broke:**  
TF-IDF treats 'Debt Collection' and 'Credit Reporting' as near-identical — both use words like 'credit', 'report', 'account'. SBERT encodes the *meaning* of the full sentence into a dense vector, capturing context that bag-of-words misses.  
The expected fix: Debt Collection F1 goes from **0.150 → 0.55+**.

**Before running:** Set runtime to T4 GPU. SBERT encoding of 140k narratives takes ~6-10 min on T4, ~40 min on CPU.

## Cell 1 — Install sentence-transformers

In [ ]:
!pip install -q sentence-transformers
print('sentence-transformers installed.')

## Cell 2 — Mount Drive & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

PROJECT_DIR = '/content/drive/MyDrive/nlp_project'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
FEATURE_DIR = os.path.join(PROJECT_DIR, 'features')
EMBED_DIR   = os.path.join(PROJECT_DIR, 'embeddings')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')
for d in [EMBED_DIR, MODEL_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

TEXT_COL  = 'narrative_clean'
LABEL_COL = 'label'

try:
    import subprocess
    subprocess.check_output('nvidia-smi', shell=True)
    DEVICE = 'cuda'
    print('GPU detected — SBERT encoding on cuda')
except:
    DEVICE = 'cpu'
    print('CPU only — encoding will take ~40 min')

## Cell 3 — Load Train / Val / Test Splits

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping  = json.load(f)
    id2label = {int(k): v for k, v in mapping['id2label'].items()}

CLASS_NAMES = [id2label[i] for i in sorted(id2label)]
N_CLASSES   = len(CLASS_NAMES)

y_train = df_train[LABEL_COL].values
y_val   = df_val[LABEL_COL].values
y_test  = df_test[LABEL_COL].values

print(f'Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')
print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')

## Cell 4 — Load SBERT Model

**Model choice: `all-MiniLM-L6-v2`**

| Property | Value |
|---|---|
| Embedding dimension | 384 |
| Layers | 6 (distilled from 12-layer BERT) |
| Max tokens | 256 |
| Model size | ~80 MB |
| Speed on T4 | ~400 sentences/sec |
| Quality | Excellent — ranks top-5 on MTEB benchmark |

We use `normalize_embeddings=True` — L2-normalises each vector to unit length.  
This makes cosine similarity equivalent to dot product and helps downstream classifiers.

In [ ]:
print('Loading all-MiniLM-L6-v2 ...')
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f'Embedding dim    : {sbert.get_sentence_embedding_dimension()}')
print(f'Max seq length   : {sbert.max_seq_length} tokens')
print(f'Device           : {DEVICE}')

## Cell 5 — Encode All Splits

We encode train, val, and test separately.  
Embeddings are saved to Drive immediately — if the session crashes, we can reload without re-encoding.

**Expected time on T4 GPU:**
- Train (139k): ~6-8 minutes
- Val + Test (~30k each): ~1-2 minutes each

The progress bar will show sentences/sec and estimated time remaining.

In [ ]:
def encode_and_save(model, df, split_name, embed_dir, text_col=TEXT_COL, batch_size=256):
    save_path = os.path.join(embed_dir, f'sbert_{split_name}.npy')
    if os.path.exists(save_path):
        emb = np.load(save_path)
        print(f'{split_name}: loaded from cache  {emb.shape}')
        return emb

    texts = df[text_col].fillna('').tolist()
    print(f'Encoding {split_name} ({len(texts):,} texts) ...')
    emb = model.encode(
        texts,
        batch_size           = batch_size,
        show_progress_bar    = True,
        normalize_embeddings = True,
        convert_to_numpy     = True,
    )
    np.save(save_path, emb)
    print(f'  Saved: {save_path}  shape={emb.shape}  size={os.path.getsize(save_path)/1e6:.1f} MB')
    return emb

emb_train = encode_and_save(sbert, df_train, 'train', EMBED_DIR)
emb_val   = encode_and_save(sbert, df_val,   'val',   EMBED_DIR)
emb_test  = encode_and_save(sbert, df_test,  'test',  EMBED_DIR)

print(f'\nAll embeddings ready.')
print(f'  emb_train : {emb_train.shape}  dtype={emb_train.dtype}')
print(f'  emb_val   : {emb_val.shape}')
print(f'  emb_test  : {emb_test.shape}')

## Cell 6 — Sanity Check: Are Embeddings Meaningful?

Before training, we verify the embeddings make semantic sense.  
We pick one complaint per class and find the most similar complaint across all classes.  
If SBERT works correctly, the nearest neighbour should be from the same class.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

print('Nearest-neighbour sanity check (same class = good embedding)\n')

# Sample one complaint per class from val set
np.random.seed(42)
for label_id in sorted(id2label):
    idx_pool = np.where(y_val == label_id)[0]
    query_idx = np.random.choice(idx_pool)
    query_emb = emb_val[query_idx:query_idx+1]

    # Find top-3 nearest neighbours (excluding self)
    sims = cosine_similarity(query_emb, emb_val)[0]
    sims[query_idx] = -1   # exclude self
    top3 = sims.argsort()[-3:][::-1]

    top3_classes = [id2label[y_val[i]] for i in top3]
    match = sum(1 for c in top3_classes if c == id2label[label_id])

    print(f'Query class : {id2label[label_id]}')
    print(f'Top-3 NN    : {top3_classes}  — {match}/3 same class')
    print()

## Cell 7 — Build Hybrid Feature Matrix

We combine the SBERT embeddings with the TF-IDF + metadata features from Notebook 03.

```
X_hybrid = [ TF-IDF (30,009) | SBERT (384) ]
                ↑ sparse          ↑ dense → converted to sparse for hstack
```

The TF-IDF part is 99.7% sparse (tiny memory).  
The SBERT part is fully dense (each of 384 dims has a value for every sample).

In [ ]:
print('Loading TF-IDF + metadata features ...')
X_tfidf_train = sp.load_npz(os.path.join(FEATURE_DIR, 'X_train.npz'))
X_tfidf_val   = sp.load_npz(os.path.join(FEATURE_DIR, 'X_val.npz'))
X_tfidf_test  = sp.load_npz(os.path.join(FEATURE_DIR, 'X_test.npz'))
print(f'TF-IDF: {X_tfidf_train.shape}')

# hstack TF-IDF (sparse) + SBERT (dense as sparse)
def build_hybrid(X_sparse, emb_dense):
    return sp.hstack(
        [X_sparse, sp.csr_matrix(emb_dense.astype(np.float32))],
        format='csr'
    )

X_hybrid_train = build_hybrid(X_tfidf_train, emb_train)
X_hybrid_val   = build_hybrid(X_tfidf_val,   emb_val)
X_hybrid_test  = build_hybrid(X_tfidf_test,  emb_test)

print(f'\nHybrid matrix shapes:')
print(f'  Train : {X_hybrid_train.shape}   (30009 TF-IDF + 384 SBERT = {X_hybrid_train.shape[1]})')
print(f'  Val   : {X_hybrid_val.shape}')
print(f'  Test  : {X_hybrid_test.shape}')

## Cell 8 — Train XGBoost on SBERT-Only Features

Training on SBERT alone (no TF-IDF) tells us exactly how much signal the embeddings add on their own.  
Expected: better macro F1 than TF-IDF alone (0.414), especially for Debt Collection.

In [ ]:
# Convert dense SBERT to sparse for XGBoost
X_sbert_train = sp.csr_matrix(emb_train.astype(np.float32))
X_sbert_val   = sp.csr_matrix(emb_val.astype(np.float32))

model_sbert = xgb.XGBClassifier(
    n_estimators          = 3000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,   # higher — only 384 features total
    min_child_weight      = 5,
    objective             = 'multi:softprob',
    num_class             = N_CLASSES,
    eval_metric           = 'mlogloss',
    device                = DEVICE,
    tree_method           = 'hist',
    random_state          = 42,
    early_stopping_rounds = 100,
)

print('Training XGBoost on SBERT-only features (384 dims) ...')
model_sbert.fit(
    X_sbert_train, y_train,
    eval_set = [(X_sbert_val, y_val)],
    verbose  = 200,
)

y_pred_sbert = model_sbert.predict(X_sbert_val)
acc_sbert    = accuracy_score(y_val, y_pred_sbert)
f1_sbert     = f1_score(y_val, y_pred_sbert, average='macro')

print(f'\n[SBERT-only]  Accuracy: {acc_sbert:.4f}  Macro F1: {f1_sbert:.4f}')
print(f'Best iteration: {model_sbert.best_iteration}')

## Cell 9 — Train XGBoost on Hybrid Features (TF-IDF + SBERT)

This is our best classical model.  
TF-IDF captures keyword frequency; SBERT captures semantic meaning.  
Together they're stronger than either alone.

`colsample_bytree=0.3` — samples ~9k features per tree from 30,393 total.  
Both TF-IDF and SBERT features get used since they're sampled proportionally.

In [ ]:
model_hybrid = xgb.XGBClassifier(
    n_estimators          = 3000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.3,
    min_child_weight      = 5,
    objective             = 'multi:softprob',
    num_class             = N_CLASSES,
    eval_metric           = 'mlogloss',
    device                = DEVICE,
    tree_method           = 'hist',
    random_state          = 42,
    early_stopping_rounds = 100,
)

print('Training XGBoost on hybrid features (TF-IDF + SBERT, 30,393 total) ...')
model_hybrid.fit(
    X_hybrid_train, y_train,
    eval_set = [(X_hybrid_val, y_val)],
    verbose  = 200,
)

y_pred_hybrid = model_hybrid.predict(X_hybrid_val)
acc_hybrid    = accuracy_score(y_val, y_pred_hybrid)
f1_hybrid     = f1_score(y_val, y_pred_hybrid, average='macro')

print(f'\n[Hybrid] Accuracy: {acc_hybrid:.4f}  Macro F1: {f1_hybrid:.4f}')
print(f'Best iteration: {model_hybrid.best_iteration}')

## Cell 10 — Model Comparison

In [ ]:
# Per-class F1 for both models
f1_sbert_pc  = f1_score(y_val, y_pred_sbert,  average=None)
f1_hybrid_pc = f1_score(y_val, y_pred_hybrid, average=None)
tfidf_f1_ref = [0.382, 0.457, 0.851, 0.150, 0.369, 0.345, 0.343]  # from NB04

print('=' * 70)
print(f'{"Class":<28} {"TF-IDF":>10} {"SBERT":>10} {"Hybrid":>10}')
print('-' * 70)
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:<28} {tfidf_f1_ref[i]:>10.3f} {f1_sbert_pc[i]:>10.3f} {f1_hybrid_pc[i]:>10.3f}')
print('-' * 70)
print(f'{"Macro F1":<28} {0.414:>10.3f} {f1_sbert:>10.3f} {f1_hybrid:>10.3f}')
print(f'{"Accuracy":<28} {0.7118:>10.3f} {acc_sbert:>10.3f} {acc_hybrid:>10.3f}')
print('=' * 70)

# Bar chart comparison
x = np.arange(len(CLASS_NAMES))
w = 0.26
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, tfidf_f1_ref,  w, label='TF-IDF',       color='steelblue',  edgecolor='white')
ax.bar(x,     f1_sbert_pc,   w, label='SBERT-only',    color='coral',      edgecolor='white')
ax.bar(x + w, f1_hybrid_pc,  w, label='Hybrid (best)', color='seagreen',   edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1: TF-IDF vs SBERT vs Hybrid (Validation Set)', fontsize=13)
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=120)
plt.show()
print('Saved: model_comparison.png')

## Cell 11 — Confusion Matrix (Hybrid Model)

In [ ]:
cm = confusion_matrix(y_val, y_pred_hybrid)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('Hybrid Model (TF-IDF + SBERT) — Normalised Confusion Matrix', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hybrid_confusion_matrix.png'), dpi=120)
plt.show()

## Cell 12 — Save Best Model

In [ ]:
# Save whichever has better macro F1
best_model = model_hybrid if f1_hybrid >= f1_sbert else model_sbert
best_name  = 'hybrid' if f1_hybrid >= f1_sbert else 'sbert_only'

path = os.path.join(MODEL_DIR, f'xgb_{best_name}.json')
best_model.save_model(path)
print(f'Best model: {best_name}')
print(f'Saved to  : {path}')
print(f'Size      : {os.path.getsize(path)/1e6:.1f} MB')

## Cell 13 — Full Classification Report (Hybrid)

In [ ]:
print('Full classification report — Hybrid model (val set):')
print(classification_report(y_val, y_pred_hybrid, target_names=CLASS_NAMES, digits=3))

## Cell 14 — Summary (paste this output to Claude)

In [ ]:
print('='*65)
print('  NOTEBOOK 05 SUMMARY — paste this output to Claude')
print('='*65)
print(f'SBERT model          : all-MiniLM-L6-v2 (384 dims)')
print(f'SBERT best iteration : {model_sbert.best_iteration}')
print(f'Hybrid best iteration: {model_hybrid.best_iteration}')
print()
print(f'{"Model":<15} {"Accuracy":>10} {"Macro F1":>10}')
print('-'*38)
print(f'{"TF-IDF (NB04)":<15} {0.7118:>10.4f} {0.4140:>10.4f}')
print(f'{"SBERT-only":<15} {acc_sbert:>10.4f} {f1_sbert:>10.4f}')
print(f'{"Hybrid":<15} {acc_hybrid:>10.4f} {f1_hybrid:>10.4f}')
print()
print('Per-class F1 — Hybrid model:')
for i, name in enumerate(CLASS_NAMES):
    delta = f1_hybrid_pc[i] - tfidf_f1_ref[i]
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '→')
    print(f'  {name:<30} {f1_hybrid_pc[i]:.3f}  {arrow} {delta:+.3f} vs TF-IDF')
print()
print(f'Best model saved     : xgb_{best_name}.json')
print('='*65)